In [1]:
# !pip install tensorboard

In [2]:
import numpy as np
import pandas as pd
from unicodedata import normalize
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [3]:
# Asegurarse de correr el script de limpieza para poder ejecutar el script a continuación
%run limpieza_data.ipynb

Todos los archivos han sido cargados
dataframe expextativas limpio
dataframe tasa_politica limpio
dataframe indice_precios limpio
dataframe tasa_ibr limpio
dataframe tasa_mercado limpio
Filtro temporal aplicado
Valores del mes seleccioandos
DATAFRAME CONSOLIDADO
Columna fecha eliminada de df_modelo_sin_fecha


## Normalización de datos

In [4]:
# Configuración de parametros para la clase BanrepDataset
n = len(df_modelo_sin_fecha)
train_obs = int(n * 0.70)
val_obs = int(n * 0.85)
test_obs = n - train_obs - val_obs

scaler = StandardScaler()
scaler.fit(df_modelo_sin_fecha[:train_obs])
df_modelo_sin_fecha = scaler.transform(df_modelo_sin_fecha)

ipc_mean = scaler.mean_[2]
ipc_std = scaler.scale_[2]

## Clase BanrepDataset

In [5]:
pasos_dato = 1
sequence_length = contexto = 12
retraso = pasos_dato * (contexto + 1 - 1)
batch_size = 32

class BanrepDataset(Dataset):
    def __init__(self, data, sequence_length, target_col, indice_inicio, indice_final, sampling_rate):
        self.data = data
        self.sequence_length = sequence_length
        self.target = target_col
        self.indices = np.arange(indice_in  icio, indice_final)
        self.sampling_rate = sampling_rate

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        inicio = self.indices[index]
        pasos = np.arange(inicio, inicio + self.sequence_length * self.sampling_rate, self.sampling_rate)

        x = self.data[pasos]
        y = self.data[inicio + retraso, self.target]
        return torch.tensor(x, dtype = torch.float32), torch.tensor(y, dtype = torch.float32)

    

## Crear Datasets

In [6]:
train_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = 0,
    indice_final = train_obs,
    sampling_rate = pasos_dato
)

val_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = train_obs - retraso,
    indice_final = val_obs - retraso,
    sampling_rate = pasos_dato
)

test_dataset = BanrepDataset(
    data = df_modelo_sin_fecha,
    sequence_length = contexto,
    target_col = 2,
    indice_inicio = val_obs - retraso,
    indice_final = n - retraso,
    sampling_rate = pasos_dato
)

## Crear dataloaders

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset,   batch_size = batch_size, shuffle = False)
test_loader = DataLoader(test_dataset,  batch_size = batch_size, shuffle = False)

# Chequeo de que los dataloaders tengan las dimensiones correctas
for entrada, vble_predecir in train_dataloader:
    print('Dimensiones entradas: ', entrada.shape)
    print('Dimensión variable objetivo: ', vble_predecir.shape)
    break

Dimensiones entradas:  torch.Size([32, 12, 5])
Dimensión variable objetivo:  torch.Size([32])


In [8]:
def correr_etapa(modelo, carga, criterio, optimizador = None):
    entrenamiento = optimizador is not None
    modelo.train() if entrenamiento else modelo.eval()
    perdida_total = 0.0
    total_mae = 0.0
    n = 0
    with torch.set_grad_enabled(entrenamiento):
        for entrada, objetivo in carga:
            entrada, objetivo = entrada.to(device), objetivo.to(device)
            predicc = modelo(entrada)
            perdida = criterio(predicc, objetivo)
            if entrenamiento:
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

            perdida_total += perdida.item() * entrada.size(0)
            total_mae += torch.sum(torch.abs(predicc - objetivo)).item()
            n += entrada.size(0)
    return perdida_total / n, total_mae / n


In [9]:
def predicciones(modelo, dataset):
    modelo.eval()
    predicciones_list = []
    objetivo_list = []
    carga = DataLoader(dataset, batch_size = 32, shuffle = False)
    with torch.no_grad():
        for entrada, objetivo in carga:
            predic = modelo(entrada.to(device)).cpu().numpy()
            predicciones_list.append(predic * ipc_std + ipc_mean)
            objetivo_list.append(objetivo.numpy())

    return np.concatenate(predicciones_list), np.concatenate(objetivo_list)

In [19]:
class punto_guardado_modelo():
    def __init__(self, filepath, monitor = 'val_mae', mode = 'min', verbose = True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float('inf') if mode == 'min' else float('-inf')
        self.mode = mode

    def step(self, metrics, model = None):
        value = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f" Mejor modelo guardado({self.monitor}: {value:.2f} Puntos)")
        
        return improved

In [20]:
class earlystopping:
    def __init__(self, monitor = 'val_mae', patience = 5, min_delta = 1e-4, mode = 'min'):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float('inf') if mode == 'min' else float('-inf')
        self.counter = 0
        self.should_stop = False

        def step(self, metrics, model = None):
            value = metrics[self.monitor]
            improved = (value < self.best - self.min_delta if self.mode == 'min'
                        else value > self.best + self.min_delta)
            if improved:
                self.bes = value
                self.counter = 0
            else:
                self.counter += 1
                if self.counter > self.patience: 
                    self.should_stop = True
                    print(f' Parada temprana debido a no mejoramiento de la metrica en {self.patience} etapas')
            return improved

In [21]:
class reduceLRonplateu:
    def __init__(self, optimizer, monitor = 'val_mae', patience = 3, factor = 0.5, verbose = True):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience = patience, factor = factor, verbose = verbose
        )
        self.monitor = monitor

        def step(self, metrics, model = None):
            self.scheduler.step(metrics[self.monitor])

In [22]:
class densemodel(nn.Module):
    def __init__(self, sequence_length, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(sequence_length * num_features, 1),
            )
    
    def forward(self, x):
        return self.network(x).squeeze(-1)

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = densemodel(sequence_length, df_modelo_sin_fecha.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer = SummaryWriter(log_dir = 'runs/modelo_ml')

callbacks = [
    punto_guardado_modelo('mejor_modelo_ML.pt', monitor = 'val_mae')
]

epocas = 10
for epoca in range(1, epocas + 1):
    train_loss, train_mae = correr_etapa(model, train_dataloader, criterion, optimizer)
    val_loss, val_mae = correr_etapa(model, val_loader, criterion)

    metrics = {'train_loss': train_loss, 'train_mae': train_mae, 'val_loss': val_loss, 'val_mae': val_mae}

    writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoca)
    writer.add_scalars('MAE', {'train': train_mae, 'val': val_mae}, epoca)

    print(f'Epoca: {epoca:02d} - '
          f'Train loss {train_loss:.04f}, Train MAE {train_mae:.2f} puntos   |'
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f} puntos")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, punto_guardado_modelo) else cb.step(metrics)

writer.close()


Epoca: 01 - Train loss 2.1589, Train MAE 1.38 puntos   |val loss: 14.9539, val MAE: 3.33 puntos
 Mejor modelo guardado(val_mae: 3.33 Puntos)
Epoca: 02 - Train loss 1.7488, Train MAE 1.24 puntos   |val loss: 11.9394, val MAE: 3.04 puntos
 Mejor modelo guardado(val_mae: 3.04 Puntos)
Epoca: 03 - Train loss 1.4077, Train MAE 1.11 puntos   |val loss: 9.3592, val MAE: 2.75 puntos
 Mejor modelo guardado(val_mae: 2.75 Puntos)
Epoca: 04 - Train loss 1.1184, Train MAE 0.99 puntos   |val loss: 7.2403, val MAE: 2.47 puntos
 Mejor modelo guardado(val_mae: 2.47 Puntos)
Epoca: 05 - Train loss 0.8909, Train MAE 0.88 puntos   |val loss: 5.5353, val MAE: 2.19 puntos
 Mejor modelo guardado(val_mae: 2.19 Puntos)
Epoca: 06 - Train loss 0.6937, Train MAE 0.77 puntos   |val loss: 4.2078, val MAE: 1.93 puntos
 Mejor modelo guardado(val_mae: 1.93 Puntos)
Epoca: 07 - Train loss 0.5385, Train MAE 0.68 puntos   |val loss: 3.1690, val MAE: 1.67 puntos
 Mejor modelo guardado(val_mae: 1.67 Puntos)
Epoca: 08 - Train 

In [24]:
model.load_state_dict(torch.load('mejor_modelo_ML.pt', map_location = device))
_, test_mae = correr_etapa(model, test_loader, criterion)
print(f'Test MAE: {test_mae:.2f}')

Test MAE: 2.35
